# OLS â€” Linear Regression (Pipeline B)

**Model**: Ordinary Least Squares via `sklearn.linear_model.LinearRegression`
**Target**: `gdpc1` (quarterly log-diff real GDP)
**Variables**: Cat2 (outbs, outnfb, gcec1, houstne + COVID dummies = 7 total)
**Train**: 1959-Q1 to 2007-Q4 / **Val**: 2008-Q1 to 2016-Q4
**Test**: 2017-Q1 to 2025-Q4 (36 quarters, m1/m2/m3 vintages)
**Scaling**: No (affine-equivariant) / **HP tuning**: No / **n_lags**: 3
**COVID injection**: Via `get_features(category="cat2", with_covid=True)` â€” passed through unscaled


In [1]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.stats import norm

plt.rcParams["figure.figsize"] = [15, 10]

sys.path.insert(0, os.path.join(os.path.pardir, "data"))
from helpers import (gen_lagged_data, gen_vintage_data, make_supervised_vintage_frame, flatten_data, mean_fill_dataset,
                     get_features, load_data,
                     PREDICTIONS_DIR, TARGET, COVID)

SEED = 42
np.random.seed(SEED)

TRAIN_START = "1959-01-01"
TRAIN_END   = "2007-12-31"
VAL_END     = "2016-12-31"
TEST_START  = "2017-01-01"
TEST_END    = "2025-12-31"
N_LAGS = 3
VINTAGES = {"m1": -2, "m2": -1, "m3": 0, "post1": 1}

print("OLS \u2014 Cat2 variables, n_lags={}, scaling=NO".format(N_LAGS))


OLS — Cat2 variables, n_lags=3, scaling=NO


In [2]:
monthly, _, metadata = load_data()
features = get_features("cat2", with_covid=True)
print("Features: {} -> {}".format(len(features), features))
print("Target: {}".format(TARGET))


Features: 7 -> ['outbs', 'outnfb', 'gcec1', 'houstne', 'covid_2020q2', 'covid_2020q3', 'covid_2020q4']
Target: gdpc1


In [3]:
test_quarters = monthly[(monthly["date"] >= TEST_START) &
                         (monthly["date"] <= TEST_END) &
                         monthly["date"].dt.month.isin([3, 6, 9, 12])]["date"].tolist()

predictions = {v: [] for v in VINTAGES}
actuals_list = []
failed = 0

for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actual = monthly[monthly["date"] == pd_q][TARGET].values[0]
    actuals_list.append(actual)

    for vintage_name, offset in VINTAGES.items():
        forecast_date = pd_q + pd.DateOffset(months=offset)
        train_flat, test_flat, _ = make_supervised_vintage_frame(
            metadata, monthly, TARGET, features, TRAIN_START, pd_q, forecast_date, N_LAGS
        )

        feature_cols = [c for c in train_flat.columns if c != "date" and c != TARGET and any(c == f or c.startswith(f + "_") for f in features)]

        if len(train_flat) < 10:
            predictions[vintage_name].append(np.nan)
            continue

        X_train = train_flat[feature_cols].values
        y_train = train_flat[TARGET].values

        try:
            model = LinearRegression()
            model.fit(X_train, y_train)

            X_test = test_flat[feature_cols].values
            pred = model.predict(X_test)[0]
        except Exception:
            pred = np.nan
            failed += 1

        predictions[vintage_name].append(pred)

    if (i + 1) % 8 == 0:
        print("  {} / {} quarters".format(i + 1, len(test_quarters)))

print("Done. {} quarters, {} failures.".format(len(test_quarters), failed))


  8 / 36 quarters


  16 / 36 quarters


  24 / 36 quarters


  32 / 36 quarters


Done. 36 quarters, 0 failures.


In [4]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
for vn in VINTAGES:
    df = pd.DataFrame({
        "date": test_quarters, "actual": actuals_list,
        "prediction": predictions[vn],
    })
    path = os.path.join(PREDICTIONS_DIR, "ols_{}.csv".format(vn))
    df.to_csv(path, index=False)
    print("Saved {}".format(path))


Saved C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\predictions\ols_m1.csv
Saved C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\predictions\ols_m2.csv
Saved C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\predictions\ols_m3.csv
Saved C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\predictions\ols_post1.csv


In [5]:
def rmse(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.sqrt(np.mean((a[m]-p[m])**2)) if m.sum()>0 else np.nan

def mae(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.mean(np.abs(a[m]-p[m])) if m.sum()>0 else np.nan

panels = {
    "Pre-COVID  (2017-2019)": ("2017-01-01", "2019-12-31"),
    "COVID      (2020-2021)": ("2020-04-01", "2021-12-31"),
    "Post-COVID (2022-2025)": ("2022-01-01", "2025-12-31"),
    "Full test  (2017-2025)": ("2017-01-01", "2025-12-31"),
}
a = np.array(actuals_list)
d = pd.to_datetime(test_quarters)
for pn, (ps, pe) in panels.items():
    m = (d >= ps) & (d <= pe)
    print("--- {} ---".format(pn))
    for vn in VINTAGES:
        pp = np.array(predictions[vn])
        print("  {}  RMSFE={:.6f}  MAE={:.6f}  N={}".format(vn, rmse(a[m],pp[m]), mae(a[m],pp[m]), m.sum()))


--- Pre-COVID  (2017-2019) ---
  m1  RMSFE=0.002736  MAE=0.002163  N=12
  m2  RMSFE=0.002728  MAE=0.002148  N=12
  m3  RMSFE=0.002662  MAE=0.002072  N=12
  post1  RMSFE=0.002626  MAE=0.002021  N=12
--- COVID      (2020-2021) ---
  m1  RMSFE=0.042756  MAE=0.026770  N=7
  m2  RMSFE=0.042704  MAE=0.026786  N=7
  m3  RMSFE=0.042641  MAE=0.026856  N=7
  post1  RMSFE=0.042604  MAE=0.026898  N=7
--- Post-COVID (2022-2025) ---
  m1  RMSFE=0.004500  MAE=0.003354  N=16
  m2  RMSFE=0.004523  MAE=0.003416  N=16
  m3  RMSFE=0.004511  MAE=0.003411  N=16
  post1  RMSFE=0.004492  MAE=0.003398  N=16
--- Full test  (2017-2025) ---
  m1  RMSFE=0.019467  MAE=0.007995  N=36
  m2  RMSFE=0.019446  MAE=0.008019  N=36
  m3  RMSFE=0.019419  MAE=0.008009  N=36
  post1  RMSFE=0.019395  MAE=0.007991  N=36
